# New Dataset Checklist
## What to Check and What to Change in the Notebook

---

## Step 1 — Run This Script on Your New Dataset First

Copy this to NERSC terminal and run it before doing anything else:

```bash
python3 << 'EOF'
import numpy as np
from PIL import Image
import os

# <- UPDATE THESE
img_dir  = "/pscratch/sd/w/worasit/datasets/rh_wheat/images"
mask_dir = "/pscratch/sd/w/worasit/datasets/rh_wheat/masks"

print("=" * 60)
print("IMAGE CHECK")
print("=" * 60)

img_files = sorted([f for f in os.listdir(img_dir) if not f.startswith('.')])
sizes = {}
rgb_count = 0
min_h, min_w = 99999, 99999

for fname in img_files:
    arr = np.array(Image.open(os.path.join(img_dir, fname)))
    if arr.ndim == 3:
        h, w = arr.shape[0], arr.shape[1]
        rgb_count += 1
    else:
        h, w = arr.shape
    sizes[arr.shape] = sizes.get(arr.shape, 0) + 1
    min_h = min(min_h, h)
    min_w = min(min_w, w)

print(f"Total images:     {len(img_files)}")
print(f"RGB images:       {rgb_count}  (rest are grayscale)")
print(f"Minimum height:   {min_h}")
print(f"Minimum width:    {min_w}")
print(f"All sizes found:")
for s, c in sorted(sizes.items()):
    print(f"  {s} → {c} images")

print()
print("=" * 60)
print("MASK CHECK")
print("=" * 60)

mask_files = sorted([f for f in os.listdir(mask_dir) if not f.startswith('.')])
all_vals = set()

for fname in mask_files:
    arr = np.array(Image.open(os.path.join(mask_dir, fname)))
    if arr.ndim == 3:
        arr = arr[:, :, 0]
    all_vals.update(np.unique(arr).tolist())

print(f"Total masks:      {len(mask_files)}")
print(f"All mask values:  {sorted(all_vals)}")
print()
print("Counts per value (from first mask):")
first_mask = np.array(Image.open(os.path.join(mask_dir, mask_files[0])))
if first_mask.ndim == 3:
    first_mask = first_mask[:, :, 0]
for v in sorted(np.unique(first_mask)):
    count = int(np.sum(first_mask == v))
    pct = count / first_mask.size * 100
    print(f"  value {v:3d}: {count:8d} pixels ({pct:.1f}%)")

print()
print("=" * 60)
print("RECOMMENDATIONS")
print("=" * 60)
safe_patch = (min(min_h, min_w) // 64) * 64
print(f"Safe PATCH_SIZE:  {safe_patch}  (must be <= {min(min_h, min_w)})")
print(f"Check if any mask value == 254 (your current ignore_index)")
print(f"If 254 is in mask values above → change ignore_index to another unused value")
EOF
```

---

## Step 2 — Answer These Questions

### A. Image Sizes
```
Q: Is minimum height/width smaller than current PATCH_SIZE (256)?
   YES -> reduce PATCH_SIZE in notebook Step 4
   NO  -> no change needed

Rule: PATCH_SIZE must be <= min(min_height, min_width) across ALL datasets
```

### B. RGB vs Grayscale
```
Q: Are there RGB images (3-channel)?
   YES -> no code change needed, already handled automatically
   NO  -> no change needed
```

### C. Mask Values
```
Q: What are the unique grayscale values in the masks?
   -> Map each value to canonical class:
     0 = Background
     1 = Epidermis
     2 = Mesophyll
     3 = Air Space
     4 = Vascular/Vein
     5 = Bundle Sheath Extensions (BSE)
```

### D. ignore_index Conflict
```
Q: Is 254 (current ignore_index) present in the mask values?
   YES -> pick any value NOT in your mask values
         e.g. if mask has [0,85,170,180,255] --> use 254 
         e.g. if mask has [0,1,2,3,4,254]    --> use 253 or 200
   NO  -> keep ignore_index = 254
```

### E. Number of Classes
```
Q: Does this dataset have BSE (Bundle Sheath Extensions)?
   YES -> include in mapping: "VALUE": 5
   NO  -> just don't include class 5 in mapping, no other changes needed
           (model will simply never see class 5 from this dataset)
```

---

## Step 3 — Create Config File

Save as `/pscratch/sd/w/worasit/configs/NEW_LAB.json`:

```json
{
  "name": "new_lab_name",
  "image_dir": "/pscratch/sd/w/worasit/datasets/new_lab/images",
  "mask_dir":  "/pscratch/sd/w/worasit/datasets/new_lab/masks",
  "mask_type": "indexed",
  "num_classes": 6,
  "ignore_index": 254,
  "class_names": [
    "background",
    "epidermis",
    "mesophyll",
    "air",
    "vein",
    "bundle_sheath"
  ],
  "mapping": {
    "THEIR_BG_VALUE":   0,
    "THEIR_EPI_VALUE":  1,
    "THEIR_MESO_VALUE": 2,
    "THEIR_AIR_VALUE":  3,
    "THEIR_VEIN_VALUE": 4,
    "THEIR_BSE_VALUE":  5
  },
  "_notes": {
    "dataset": "Brief description",
    "facility": "Lab name / beamline",
    "mask_values_verified": "list values you confirmed",
    "classes_present": ["list which classes exist"],
    "classes_absent": ["list which classes are missing"]
  }
}
```

---

## Step 4 — Changes in Notebook

### Only these things may need changing:
```  
| What           | Where in Notebook | Change When                                     |
|----------------|-------------------|-------------------------------------------------|
|  CONFIG_PATHS  | Step 4            | **Always** — add new config path                |
|  PATCH_SIZE    | Step 4            | New dataset has smaller images than current min |
|  STRIDE        | Step 4            | Usually keep at PATCH_SIZE / 2                  |
|  IGNORE_INDEX  | Step 4            | New dataset has 254 as a mask value             |
|  BATCH_SIZE    | Step 4            | OOM error on GPU                                |
```

### Everything else is automatic:
```
RGB handling      -> automatic in LeafDataset 
Size variation    -> automatic in PatchDataset
Missing classes   -> automatic (just not in mapping)
Different values  -> handled by config mapping
Padding           -> automatic in PatchDataset
```

### Example — Adding a new dataset:
```python
# Step 4 in notebook — ONLY change is adding one line:
CONFIG_PATHS = [
    f"{SCRATCH}/configs/devin1.json",
    f"{SCRATCH}/configs/arabidopsis.json",   # <- just add this!
]

# If new dataset has min height = 200:
PATCH_SIZE = 192   # <- reduce this too (must be <= 200, use multiple of 64)
STRIDE = 96
```

---

## Reference: Known Datasets
```
| Dataset                  | Mask Values             | Classes    | Ignore_index | Min Size |
|--------------------------|-------------------------|------------|--------------|----------|
| devin1                   | 0,85,170,180,255        | 5 (no BSE) | 254          | 323×734  |
| devin1                   | 0,85,152,170,180,255    | 6          | -            | 323×734  |
| devin2                   | 0,85,95,152,170,180,255 |            | 254          | 578×1436 | *95-Epidermis
| devin3                   | 0,85,103,170,255        |            | -            | 501×1503 | *103-Vein
*(add new datasets here as you confirm them)* 
```
---

## Canonical Class Reference
```
Index | Class Name          | Typical grayscale (varies by lab!)
------+---------------------+-----------------------------------
  0   | Background          | varies
  1   | Epidermis           | varies
  2   | Mesophyll           | varies
  3   | Air Space (IAS)     | varies
  4   | Vascular / Vein     | varies
  5   | Bundle Sheath (BSE) | varies — NOT in all datasets
```

**Rule:** The grayscale VALUE changes between labs. The canonical INDEX (0-5) never changes. The config mapping bridges the two.
